# MyDigitalTwin — Spotify
**Notebook — Ingestion incrémentale → Warehouse Delta Lake**

Sources (`data/processed/SPOTIFY/`) :
- `account/StreamingHistory_music_*.json` — historique récent, ~1 an (format court)
- `extended/Streaming_History_Audio_*.json` — historique complet depuis 2020 (format riche)
- `account/YourLibrary.json` — titres likés (identité musicale forte)
- `account/Playlist1.json` — playlists personnelles
- `account/SearchQueries.json` — recherches Spotify

Outputs (warehouse Delta Lake) :
- `spotify_streams` — écoutes nettoyées (>30s), fusion Extended+Account, déduplication
- `spotify_liked_songs` — titres likés avec `trackUri` (pour graphe topologique)
- `spotify_playlists` — tracks de toutes les playlists
- `spotify_searches` — requêtes de recherche

## Stratégie d'ingestion incrémentale
| Source | Couverture | Schéma |
|---|---|---|
| Extended | 2020-11-21 → 2025-05-03 | Riche (`trackUri`, `skipped`, `shuffle`) |
| Account  | 2025-03-29 → 2026-03-30 | Simple (`endTime`, `msPlayed`) |

**Principe** : union des deux sources → déduplication par clé `(artistName, trackName, minute)` en favorisant Extended → écriture Delta `overwrite` (idempotent).

Lors des prochains exports, les nouveaux fichiers s'accumulent dans `processed/` (parser OVERWRITE=False).  
Il suffit de relancer ce notebook pour intégrer les nouveaux événements sans doublons.

## 0. Initialisation Spark

In [ ]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import PROCESSED_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType
import glob

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Spotify") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"PROCESSED_DATA : {PROCESSED_DATA}")
print(f"WAREHOUSE      : {WAREHOUSE}")

## 1. Chemins & Configuration

In [ ]:
SPOTIFY_ACCOUNT  = os.path.join(PROCESSED_DATA, "SPOTIFY", "account")
SPOTIFY_EXTENDED = os.path.join(PROCESSED_DATA, "SPOTIFY", "extended")

LIBRARY_PATH      = os.path.join(SPOTIFY_ACCOUNT, "YourLibrary.json")
PLAYLIST_PATH     = os.path.join(SPOTIFY_ACCOUNT, "Playlist1.json")
SEARCH_PATH       = os.path.join(SPOTIFY_ACCOUNT, "SearchQueries.json")

OUT_STREAMS       = os.path.join(WAREHOUSE, "spotify_streams")
OUT_LIKED_SONGS   = os.path.join(WAREHOUSE, "spotify_liked_songs")
OUT_PLAYLISTS     = os.path.join(WAREHOUSE, "spotify_playlists")
OUT_SEARCHES      = os.path.join(WAREHOUSE, "spotify_searches")

# Lister les fichiers disponibles
extended_files = sorted(glob.glob(os.path.join(SPOTIFY_EXTENDED, "Streaming_History_Audio_*.json")))
account_files  = sorted(glob.glob(os.path.join(SPOTIFY_ACCOUNT,  "StreamingHistory_music_*.json")))

print(f"Extended History : {len(extended_files)} fichiers")
for f in extended_files:
    print(f"  {os.path.basename(f)}")
print(f"\nAccount Data     : {len(account_files)} fichiers")
for f in account_files:
    print(f"  {os.path.basename(f)}")

## 2. Extended Streaming History

Source principale : 19 fichiers JSON depuis 2020.  
Schéma riche : `ts`, `ms_played`, `master_metadata_*`, `spotify_track_uri`, `skipped`, `shuffle`.

In [ ]:
df_ext_raw = spark.read \
    .option("multiLine", "true") \
    .json(extended_files)

print(f"Lignes brutes Extended : {df_ext_raw.count():,}")
print("Schema :")
df_ext_raw.printSchema()

# Normalisation vers schéma unifié
# ts format : "2020-11-21T12:47:13Z" (ISO 8601 UTC)
df_ext = df_ext_raw \
    .filter(F.col("master_metadata_track_name").isNotNull()) \
    .select(
        F.to_timestamp(F.col("ts"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("listen_ts"),
        F.col("master_metadata_album_artist_name").alias("artistName"),
        F.col("master_metadata_track_name").alias("trackName"),
        F.col("ms_played").cast(LongType()).alias("msPlayed"),
        F.col("spotify_track_uri").alias("trackUri"),
        F.col("skipped").cast(BooleanType()).alias("skipped"),
        F.col("shuffle").cast(BooleanType()).alias("shuffle"),
        F.lit("extended").alias("_source"),
    ) \
    .filter(F.col("listen_ts").isNotNull())

print(f"\nLignes Extended normalisées (tracks uniquement) : {df_ext.count():,}")
df_ext.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier")
).show(truncate=False)

## 3. Account Data Streaming History

Source complémentaire : couvre la période **2025-05-03 → 2026-03-30** non présente dans Extended.  
Schéma simple : `endTime` (minute), `artistName`, `trackName`, `msPlayed`.

In [ ]:
df_acc_raw = spark.read \
    .option("multiLine", "true") \
    .json(account_files)

print(f"Lignes brutes Account Data : {df_acc_raw.count():,}")

# Normalisation vers le même schéma unifié
# endTime format : "2025-03-29 17:55" (UTC, précision minute)
df_acc = df_acc_raw \
    .filter(F.col("trackName").isNotNull()) \
    .select(
        F.to_timestamp(F.col("endTime"), "yyyy-MM-dd HH:mm").alias("listen_ts"),
        F.col("artistName"),
        F.col("trackName"),
        F.col("msPlayed").cast(LongType()),
        F.lit(None).cast(StringType()).alias("trackUri"),
        F.lit(None).cast(BooleanType()).alias("skipped"),
        F.lit(None).cast(BooleanType()).alias("shuffle"),
        F.lit("account").alias("_source"),
    ) \
    .filter(F.col("listen_ts").isNotNull())

print(f"Lignes Account normalisées : {df_acc.count():,}")
df_acc.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier")
).show(truncate=False)

## 4. Fusion & Déduplication (ingestion incrémentale)

**Clé de déduplication** : `(artistName, trackName, minute)` où `minute = date_trunc("minute", listen_ts)`.

En cas de doublon (même écoute présente dans Extended ET Account), on conserve la version **Extended** (schéma plus riche : `trackUri`, `skipped`, `shuffle`).  
Les écoutes uniques à Account (2025-05-03 → 2026-03-30) sont toutes conservées.

Ce mécanisme est **idempotent** : relancer le notebook avec les mêmes fichiers donne exactement le même résultat.

In [ ]:
# Union des deux sources
df_all = df_ext.union(df_acc)

print(f"Total avant déduplication : {df_all.count():,}")
df_all.groupBy("_source").count().orderBy("_source").show()

# Colonnes helper pour la déduplication
# _dedup_min : clé à la minute (Extended = seconde, Account = minute → même granularité)
# _priority  : 0 pour Extended (gardé), 1 pour Account (écarté si doublon)
df_all = df_all \
    .withColumn(
        "_dedup_min",
        F.date_format(F.date_trunc("minute", F.col("listen_ts")), "yyyy-MM-dd HH:mm")
    ) \
    .withColumn(
        "_priority",
        F.when(F.col("_source") == "extended", 0).otherwise(1)
    )

# Window : pour chaque (artist, track, minute), garder la ligne de plus haute priorité
w = Window \
    .partitionBy("artistName", "trackName", "_dedup_min") \
    .orderBy("_priority")

df_merged = df_all \
    .withColumn("_rn", F.row_number().over(w)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn", "_source", "_dedup_min", "_priority")

total_merged = df_merged.count()
print(f"\nTotal après déduplication : {total_merged:,}")
print(f"Doublons supprimés        : {df_all.count() - total_merged:,}")
df_merged.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts")
).show(truncate=False)

## 5. Nettoyage & Features temporelles

Règles :
- **Filtre >30s** : `msPlayed >= 30 000` — élimine les skips et pré-écoutes
- **Features temporelles** : heure, jour, mois, année, semaine
- **`is_night`** : 22h–5h
- **`interaction_weight`** : 1.0 (poids ALS standard pour les streams)

In [ ]:
# Filtre écoutes réelles (>30s)
df_streams = df_merged.filter(F.col("msPlayed") >= 30000)
print(f"Lignes après filtre >30s : {df_streams.count():,}")

# Features temporelles
df_streams = df_streams \
    .withColumn("listen_year",    F.year("listen_ts")) \
    .withColumn("listen_month",   F.date_format("listen_ts", "yyyy-MM")) \
    .withColumn("listen_hour",    F.hour("listen_ts")) \
    .withColumn("listen_weekday", F.dayofweek("listen_ts")) \
    .withColumn("listen_week",    F.weekofyear("listen_ts")) \
    .withColumn("minutes_played", F.round(F.col("msPlayed") / 60000.0, 2))

df_streams = df_streams.withColumn(
    "is_night",
    F.when((F.col("listen_hour") >= 22) | (F.col("listen_hour") <= 5), True).otherwise(False)
)

df_streams = df_streams.withColumn("interaction_weight", F.lit(1.0))

# Schema final
df_streams_final = df_streams.select(
    "artistName",
    "trackName",
    "msPlayed",
    "minutes_played",
    "trackUri",
    "skipped",
    "shuffle",
    "listen_ts",
    "listen_year",
    "listen_month",
    "listen_hour",
    "listen_weekday",
    "listen_week",
    "is_night",
    "interaction_weight",
)

print(f"Lignes finales : {df_streams_final.count():,}")
df_streams_final.printSchema()
df_streams_final.show(5, truncate=45)

## 6. Exploration

In [ ]:
print("=== Période couverte ===")
df_streams_final.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts"),
    F.count(F.when(F.col("trackUri").isNotNull(), 1)).alias("avec_uri")
).show(truncate=False)

print("\n=== Écoutes par année ===")
df_streams_final.groupBy("listen_year") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 0).alias("total_minutes")
    ) \
    .orderBy("listen_year") \
    .show()

print("\n=== Top 20 artistes ===")
df_streams_final.groupBy("artistName") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 1).alias("total_minutes")
    ) \
    .orderBy(F.desc("nb_ecoutes")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Écoutes par heure ===")
df_streams_final.groupBy("listen_hour") \
    .count() \
    .orderBy("listen_hour") \
    .show(24)

print("\n=== % écoutes nocturnes ===")
total = df_streams_final.count()
night = df_streams_final.filter(F.col("is_night")).count()
print(f"Nuit : {night:,} / {total:,} ({night/total*100:.1f}%)")

## 7. Écriture spotify_streams (Delta, overwrite idempotent)

In [ ]:
df_streams_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_STREAMS)

print(f"Ecrit : {OUT_STREAMS}")
df_check = spark.read.format("delta").load(OUT_STREAMS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

---
## 8. YourLibrary — Titres likés

Titres sauvegardés = identité musicale forte.  
Signal ALS fort → **poids 3.0**.  
La colonne `trackUri` est utilisée pour le graphe topologique (Phase 2G).

In [ ]:
df_lib_raw = spark.read \
    .option("multiLine", "true") \
    .json(LIBRARY_PATH)

df_lib_raw.printSchema()

# Explosion du tableau 'tracks'
df_liked = df_lib_raw \
    .select(F.explode("tracks").alias("t")) \
    .select(
        F.col("t.artist").alias("artistName"),
        F.col("t.album").alias("albumName"),
        F.col("t.track").alias("trackName"),
        F.col("t.uri").alias("trackUri")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(3.0))

print(f"Titres likés : {df_liked.count():,}")
df_liked.show(10, truncate=45)

### Écriture spotify_liked_songs

In [ ]:
df_liked.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_LIKED_SONGS)

print(f"Ecrit : {OUT_LIKED_SONGS}")
df_check = spark.read.format("delta").load(OUT_LIKED_SONGS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

---
## 9. Playlists

Tracks ajoutés manuellement = curation active.  
Signal ALS intermédiaire → **poids 2.0**.

In [ ]:
df_pl_raw = spark.read \
    .option("multiLine", "true") \
    .json(PLAYLIST_PATH)

# Explosion playlists -> items -> track
df_playlists = df_pl_raw \
    .select(F.explode("playlists").alias("pl")) \
    .select(
        F.col("pl.name").alias("playlistName"),
        F.col("pl.lastModifiedDate").alias("lastModifiedDate"),
        F.explode("pl.items").alias("item")
    ) \
    .select(
        F.col("playlistName"),
        F.col("lastModifiedDate"),
        F.col("item.track.trackName").alias("trackName"),
        F.col("item.track.artistName").alias("artistName"),
        F.col("item.track.albumName").alias("albumName"),
        F.col("item.track.trackUri").alias("trackUri"),
        F.to_date(F.col("item.addedDate"), "yyyy-MM-dd").alias("addedDate")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(2.0))

print(f"Tracks totaux dans les playlists : {df_playlists.count():,}")
df_playlists.groupBy("playlistName", "lastModifiedDate") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(35, truncate=50)

### Écriture spotify_playlists

In [ ]:
df_playlists.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_PLAYLISTS)

print(f"Ecrit : {OUT_PLAYLISTS}")
df_check = spark.read.format("delta").load(OUT_PLAYLISTS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

---
## 10. Recherches — spotify_searches

2 483 requêtes de recherche depuis `SearchQueries.json`.  
Format : `{ platform, searchTime, searchQuery, searchInteractionURIs }`

In [ ]:
df_search_raw = spark.read \
    .option("multiLine", "true") \
    .json(SEARCH_PATH)

print(f"Recherches brutes : {df_search_raw.count():,}")
df_search_raw.printSchema()
df_search_raw.show(5, truncate=60)

# Normalisation
# searchTime format : "2025-02-07T11:07:53.687Z[UTC]" — strip [UTC] suffix
df_searches = df_search_raw \
    .filter(F.col("searchQuery").isNotNull() & (F.length(F.col("searchQuery")) > 0)) \
    .select(
        F.to_timestamp(
            F.regexp_replace(F.col("searchTime"), r"\[UTC\]$", ""),
            "yyyy-MM-dd'T'HH:mm:ss.SSSX"
        ).alias("search_ts"),
        F.trim(F.col("searchQuery")).alias("query"),
        F.col("platform")
    ) \
    .filter(F.col("search_ts").isNotNull()) \
    .withColumn("event_hour",    F.hour("search_ts")) \
    .withColumn("event_weekday", F.dayofweek("search_ts"))

print(f"\nRecherches nettoyées : {df_searches.count():,}")
df_searches.agg(
    F.min("search_ts").alias("premier"),
    F.max("search_ts").alias("dernier")
).show(truncate=False)
df_searches.show(10, truncate=55)

### Écriture spotify_searches

In [ ]:
df_searches.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_SEARCHES)

print(f"Ecrit : {OUT_SEARCHES}")
df_check = spark.read.format("delta").load(OUT_SEARCHES)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=55)

---
## Récapitulatif

In [ ]:
tables = [
    ("spotify_streams",    OUT_STREAMS),
    ("spotify_liked_songs", OUT_LIKED_SONGS),
    ("spotify_playlists",  OUT_PLAYLISTS),
    ("spotify_searches",   OUT_SEARCHES),
]

print("\n" + "="*55)
print("  Tables Spotify ecrites dans le warehouse")
print("="*55)
for name, path in tables:
    try:
        n = spark.read.format("delta").load(path).count()
        print(f"  {name:<25} {n:>8,} lignes")
    except Exception as e:
        print(f"  {name:<25} ERREUR: {e}")
print("="*55 + "\n")

In [ ]:
spark.stop()